In [3]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import tensorflow as tf

from tensorflow.keras import layers, models

import pandas as pd

In [ ]:


# 1. Load pre-trained tokenizer and model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\anaconda3_5\envs\envtda\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


tensor([1, 1])


In [14]:

# 2. Prepare your data
# Example: using a custom dataset (replace this with your own data)
df=pd.read_csv('../datos/train_poll_v1s1.csv')
texts = df['respuestas'].tolist()
labels = df['ai'].tolist()

# Take only teh first 100 samples
texts = texts[:100]
labels = labels[:100]

# Separate by train and test
train_texts = texts[:int(len(texts)*0.8)]
train_labels = labels[:int(len(labels)*0.8)]
test_texts = texts[int(len(texts)*0.8):]
test_labels = labels[int(len(labels)*0.8):]

In [13]:

# Tokenize
encodings = tokenizer(train_texts, truncation=True, padding=True, return_tensors="pt")

# Create a dataset object
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

dataset = SimpleDataset(encodings, train_labels)

# 3. Set up Trainer
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_dir="./logs",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# 4. Train
trainer.train()

# 5. Predict
test_texts = ["I really like this product.", "I don't like this."]
test_encodings = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt")
outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=1)
print(predictions)  # 1 for positive, 0 for negative


d:\anaconda3_5\envs\envtda\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.642000
20,0.432700
30,0.194700
40,0.012900
50,0.075400
60,0.142800


tensor([0, 0])


In [17]:
# Predict on test set
test_encodings = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt")
test_dataset = SimpleDataset(test_encodings, test_labels)
a=trainer.evaluate(test_dataset)

d:\anaconda3_5\envs\envtda\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [24]:
outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=1)
print(predictions)  # 1 for positive, 0 for negative
print(test_labels)  # 1 for positive, 0 for negative
# print the accuracy



tensor([0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0])
[0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0]


In [25]:
predictions =list(predictions.numpy())
test_labels = list(test_labels)
# print the accuracy
correct = 0
for i in range(len(predictions)):
    if predictions[i] == test_labels[i]:
        correct += 1
accuracy = correct / len(predictions)
print("Accuracy: ", accuracy)

Accuracy:  1.0


In [27]:
for i in range(len(test_texts)):
    print(test_texts[i], " : ", predictions[i], " : ", test_labels[i])

Que se vea que le apasione el tema y que pueda explicar temas difíciles con palabras sencillas   :  0  :  0
La unidad de medida de un segundo se define como la duración de 9.192.631.770 vibraciones del átomo de cesio-133. Esta se utiliza principalmente para medir el tiempo en diversas aplicaciones y campos, como en la física, la tecnología, la aviación, la navegación y en la industria de relojería, entre otros. También se utiliza en experimentos científicos para medir con precisión el tiempo de reacciones químicas, así como en la medicina para medir la frecuencia cardíaca y el tiempo de tratamiento de algunas enfermedades.  :  1  :  1
El día de la raza, porque es el día en que Cristóbal Colón llegó a América   :  0  :  0
-tareas realizadas
-evaluación de conocimientos 
-proyecto
-disposicion de aprender
-participaciónes 
-compromiso con el curso  :  0  :  0
Que no esté seco, que tenga un buen balances de sabores, ni muy dulce, ni muy amargo. Que no exageren con el betún. Que este bien 